# PDF 파일을 사용한 RAG 구현

In [1]:
# !pip install pymupdf

In [2]:
from dotenv import load_dotenv
import os

import pymupdf as fitz

# .env 파일 불러오기
load_dotenv("C:/env/.env")

# 환경 변수 가져오기
API_KEY = os.getenv("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI(api_key=API_KEY)

#### PDF 파일 텍스트 추출 및 요약 

In [3]:
#  PDF 파일 텍스트 추출 함수
def extract_text_from_pdf(pdf_path):
    text = ""
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text += page.get_text()
    return text        

# OpenAI API 호출 텍스트 요약
def summarize_text(text):
    prompt = f"다음 문서를 요약해줘:\n\n{text}"
    
    response = client.chat.completions.create(
        model = "gpt-4o-mini", 
        messages = [ {"role":"user","content":prompt} ],
        temperature = 0.2
    )    

    return response.choices[0].message.content    

In [4]:
# PDF 파일 텍스트 추출
pdf_path = "기상조건에 따라 고도가 강수 특성에 미치는 영향_ 제주도 사례 분석.pdf"
full_text = extract_text_from_pdf(pdf_path)
print(full_text)

# 읽어온 PDF 텍스트를 텍스트 파일로 저장
txt_output_path = os.path.splitext(pdf_path)[0] + "_fulltext.txt"
with open(txt_output_path, "w", encoding="utf-8") as f:
    f.write(full_text)
print("텍스트 추출 및 저장 완료!!\n")

# 문서 요약
summary = summarize_text(full_text)
print("요약 결과:\n")
print(summary)

# 요약 결과를 텍스트 파일로 저장
summary_output_path = os.path.splitext(pdf_path)[0] + "_summary.txt"
with open(summary_output_path, "w", encoding="utf-8") as f:
    f.write(summary)  
print("텍스트 요약 및 저장 완료!!\n")

Atmosphere. Korean Meteorological Society
Vol. 35, No. 3 (2025) pp. 369-384
https://doi.org/10.14191/Atmos.2025.35.3.369
pISSN 1598-3560
eISSN 2288-3266
 2025 Korean Meteorological Society
369
기상조건에 따라 고도가 강수 특성에 미치는 영향: 제주도 사례 분석
이현정1),2) · 서명석1),2)*
1)국립공주대학교 대기과학과, 2)기상기후데이터 융합 분석 특성화 대학원
(접수일: 2025년 5월 21일, 수정일: 2025년 7월 3일, 게재확정일: 2025년 7월 12일)
The Influence of Altitude on Precipitation Characteristics Based
on Meteorological Conditions: A Case Study of Jeju Island
Hyeon-Jeong Lee1),2) and Myoung-Seok Suh1),2)*
1)Department of Atmospheric Science, Kongju National University, Gongju, Korea
2)Specialized Graduate School for Integrated Analysis of Meteorological and Climatic Data
(Manuscript received 21 May 2025; revised 3 July 2025; accepted 12 July 2025)
Abstract
In this study, the influence of altitude (Inf_o_Alt) on precipitation characteristics
(amount: Pr_Amt, frequency: Pr_Fre, intensity: Pr_Int) was investigated over various time
scales and meteorological conditions (wet/dry

### PDF 파일을 사용한 RAG 구현

In [5]:
from dotenv import load_dotenv
import os

import pymupdf as fitz

import faiss
import numpy as np
import pickle

# .env 파일 불러오기
load_dotenv("C:/env/.env")

# 환경 변수 가져오기
API_KEY = os.getenv("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI(api_key=API_KEY)

In [5]:
# 1) PDF 파일 텍스트 추출 함수
def extract_text_from_pdf(pdf_path):
    text = ""
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text += page.get_text()
    return text   

# 2) 텍스트를 청크로 나누는 함수
def split_text_into_chunks(text,chunk_size=1000,overlap=200):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap   # 겹치는 부분을 두어 문맥 연결성 유지

    return chunks

In [6]:
# 3) OpenAI 임베딩 생성 함수 : 텍스트 리스트를 임베딩 벡터로 변환
def get_embeddings(texts):

    embeddings = []

    for text in texts:
        response = client.embeddings.create(
            model = 'text-embedding-3-small', 
            input = text        
        )
        embedding = response.data[0].embedding
        embeddings.append(embedding)
        
    return np.array(embeddings,dtype=np.float32)

# 4) FAISS 인덱스 생성 및 저장, 청크 저장 함수
def create_faiss_index_chunk(embeddings,chunks,index_path="faiss_index.bin",chunks_path="chunks.pkl"):

    # FAISS 인덱스 생성 (L2 거리 기반)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)

    # 임베딩을 인덱스에 추가
    index.add(embeddings)

    # 인덱스와 청크 저장
    faiss.write_index(index,index_path)
    with open(chunks_path,'wb') as f:
        pickle.dump(chunks,f)

    print(f"FAISS 인덱스가 '{index_path}'에 저장되었습니다.")
    print(f"텍스트 청크가 '{chunks_path}'에 저장되었습니다.")
    
    return index

In [7]:
# 5) FAISS 인덱스와 청크 로드 함수
def load_faiss_index_chunk(index_path="faiss_index.bin",chunks_path="chunks.pkl"):
    index = faiss.read_index(index_path)
    with open(chunks_path,'rb') as f:
        chunks = pickle.load(f)

    return index,chunks

# 6) 질의에 대한 유사도 검색 함수  : 질의와 유사한 청크를 검색
def search_similar_chunks(query,index,chunks,top_k=3):

    # 질의를 임베딩으로 변환
    query_embedding = get_embeddings([query])

    # FAISS에서 유사한 벡터 검색
    distances, indices = index.search(query_embedding,top_k)

    # 검색된 청크들 반환
    similar_chunks = []
    for i, idx in enumerate(indices[0]):
        similar_chunks.append({
            'chunk': chunks[idx],
            'distance': distances[0][i]            
        })
    return similar_chunks

In [8]:
# 7) RAG 기반 답변 생성 함수
def generate_rag_answer(query,similar_chunks):

    # 컨텍스트 구성
    context = "\n\n".join([chunk['chunk'] for chunk in similar_chunks])

    # 프롬프트 구성
    prompt = f"""다음 문서 내용을 참고하여 질문에 답변해주세요.

    문서 내용:
    {context}
    
    질문: {query}
    
    답변:"""

    # OpenAI API 호출
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[ {"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=1000
    )
    
    return response.choices[0].message.content    

In [9]:
# 8) 전체 RAG 시스템 실행 함수
def run_pdf_rag(pdf_path,query,rebuild_index=False):
    
    index_path = "faiss_index.bin"
    chunks_path = "chunks.pkl"

    # 인덱스가 없거나 재구축이 필요한 경우
    if rebuild_index or not os.path.exists(index_path):

        print("--> PDF에서 텍스트 추출 중...")
        full_text = extract_text_from_pdf(pdf_path)

        print("--> 텍스트를 청크로 분할 중...")
        chunks = split_text_into_chunks(full_text)

        print("--> 임베딩 생성 중...")
        embeddings = get_embeddings(chunks)

        print("--> FAISS 인덱스 생성 및 저장 중...")
        index = create_faiss_index_chunk(embeddings,chunks,index_path,chunks_path)

    else: # 인덱스가 있을 경우        
        print("--> 기존 FAISS 인덱스 로드 중...")
        index,chunks = load_faiss_index_chunk(index_path,chunks_path)

    # 질의 처리
    print(f"\n🔍 질의: {query}")
    print("🔎 유사한 내용 검색 중...")
    similar_chunks = search_similar_chunks(query,index,chunks)

    print("🤖 답변 생성 중...")
    answer = generate_rag_answer(query,similar_chunks)

    return answer, similar_chunks

In [10]:
# 9) 실행 
if __name__ == '__main__':

    # PDF 파일 경로
    pdf_path = '기상조건에 따라 고도가 강수 특성에 미치는 영향_ 제주도 사례 분석.pdf'

    # 질의 예시들
    queries = [
        "제주도의 강수 특성은 어떻게 나타나나요?",
        "고도가 강수에 미치는 영향은 무엇인가요?",
        "이 연구의 주요 결론은 무엇인가요?"
    ]    

    # 첫 번째 실행 시에는 인덱스를 새로 만들고, 이후에는 기존 인덱스 사용
    for i,query in enumerate(queries):
        print(f"\n{'='*60}")
        print(f"질의 {i+1}: {query}")
        print('='*60)

        # 첫 번째 질의에서만 인덱스 재구축
        rebuild = (i == 0)

        try:
            answer, similar_chunks = run_pdf_rag(pdf_path, query, rebuild_index=rebuild)
            
            print(f"\n💡 답변:\n{answer}")
            
            print(f"\n📋 참조된 문서 내용 (상위 {len(similar_chunks)}개):")
            for j, chunk in enumerate(similar_chunks):
                print(f"\n[참조 {j+1}] (유사도: {chunk['distance']:.4f})")
                print(f"{chunk['chunk'][:200]}...")
                
        except Exception as e:
            print(f"오류 발생: {e}")


질의 1: 제주도의 강수 특성은 어떻게 나타나나요?
--> PDF에서 텍스트 추출 중...
--> 텍스트를 청크로 분할 중...
--> 임베딩 생성 중...
--> FAISS 인덱스 생성 및 저장 중...
FAISS 인덱스가 'faiss_index.bin'에 저장되었습니다.
텍스트 청크가 'chunks.pkl'에 저장되었습니다.

🔍 질의: 제주도의 강수 특성은 어떻게 나타나나요?
🔎 유사한 내용 검색 중...
🤖 답변 생성 중...

💡 답변:
제주도의 강수 특성은 고도에 따라 매우 상이하게 나타납니다. 저지대(250 m 이하)에서는 연평균 강수량이 약 2,500 mm인 반면, 고지대(1,500 m 이상)에서는 6,000~7,000 mm에 달합니다. 이는 제주도가 남한에서 해발고도가 가장 높은 한라산을 중심으로 위치하고 있으며, 동아시아 몬순의 영향으로 계절에 따라 기류가 달라지고 다양한 고도가 존재하기 때문입니다.

강수량은 주로 여름에 가장 강한 영향을 받으며, 강수 빈도는 새벽에 최대치를 보이는 경향이 있습니다. 강수 강도는 분석 자료에 따라 차이가 있지만, 새벽과 오후에 두 번의 최대치를 나타내는 쌍봉형 패턴을 보입니다. 고도가 강수량에 미치는 영향은 주로 새벽과 늦은 오후 시간대에 두 번의 최대치를 보이며, 강수 빈도와 강도는 서로 다른 패턴을 보입니다.

또한, 제주도에서는 풍상측과 풍하측에서 고도가 강수 특성에 미치는 영향의 차이가 크지 않은 것으로 나타났으며, 이는 한라산의 원추형 구조와 관련이 있습니다. 제주도는 남쪽 또는 남서쪽에서 접근하는 저기압의 영향을 받아 온난 습윤한 공기가 유입될 때 고도의 영향이 강하게 작용하는 것으로 보입니다. 이러한 특성들은 제주도의 복잡한 지형과 기상 조건에 기인합니다.

📋 참조된 문서 내용 (상위 3개):

[참조 1] (유사도: 0.8564)
de, Precipitation characteristics, Meteorological conditions, Jeju Island
1. 서
론
우리나라는 삼면이